# Supervised Reinforcement Learning (SRL) with GRPOTrainer

This notebook demonstrates **Supervised Reinforcement Learning (SRL)** - a step-level training approach for reasoning models using TRL's GRPOTrainer.

**What is SRL?**
- Instead of training on complete solutions, SRL trains at each reasoning step
- Creates training pairs: `Q + S1 → predict S2`, `Q + S1 + S2 → predict S3`, etc.
- Uses GRPO's group comparison at the step level

**Full implementation:** [GitHub Repository](https://github.com/YOUR_USERNAME/Supervised-Reinforcement-Learning)

## 1. Install Dependencies

In [ ]:
%%capture
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git
!pip install trl datasets

## 2. Load Model with Unsloth + vLLM

In [ ]:
from unsloth import FastLanguageModel, PatchFastRL

# Patch TRL for Unsloth compatibility
PatchFastRL("GRPO", FastLanguageModel)

# Load model with vLLM for fast inference
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
    fast_inference=True,  # Enables vLLM with sleep mode
    gpu_memory_utilization=0.6,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    use_gradient_checkpointing="unsloth",
)

## 3. Create SRL Dataset

SRL transforms Chain-of-Thought data into step-level training pairs.

In [ ]:
from datasets import Dataset

# SRL System Instruction
SRL_INSTRUCTION = """You are a step-by-step reasoning assistant.
Given the current state of reasoning, provide the next logical step.
Focus on making one clear, correct step forward."""

# Example SRL data (in practice, load from JSONL)
srl_samples = [
    {
        "input_prompt": "Problem: If x + 5 = 12, what is x?\n\nStep 1: Subtract 5 from both sides.",
        "expert_action": "Step 2: x + 5 - 5 = 12 - 5, so x = 7."
    },
    {
        "input_prompt": "Problem: Calculate 15% of 80.\n\nStep 1: Convert 15% to decimal: 0.15",
        "expert_action": "Step 2: Multiply 0.15 × 80 = 12."
    },
    {
        "input_prompt": "Problem: Find the area of a triangle with base 6 and height 4.",
        "expert_action": "Step 1: Use formula A = (1/2) × base × height."
    },
]

# Convert to prompts with chat template
def create_srl_prompt(sample):
    messages = [
        {"role": "system", "content": SRL_INSTRUCTION},
        {"role": "user", "content": sample["input_prompt"]}
    ]
    return {
        "prompt": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True),
        "expert_action": sample["expert_action"]
    }

train_dataset = Dataset.from_list([create_srl_prompt(s) for s in srl_samples])
print(f"Dataset size: {len(train_dataset)}")
print(f"Sample prompt:\n{train_dataset[0]['prompt'][:300]}...")

## 4. Define SRL Reward Function

Compares generated step with expert action using semantic similarity.

In [ ]:
from difflib import SequenceMatcher

def srl_reward_function(prompts, completions, expert_action, **kwargs):
    """
    SRL reward: Compare generated step with expert action.
    
    Returns higher reward for more similar outputs.
    """
    rewards = []
    for completion, expert in zip(completions, expert_action):
        # Extract generated text
        if isinstance(completion, list) and len(completion) > 0:
            generated = completion[0].get("content", "") if isinstance(completion[0], dict) else str(completion[0])
        else:
            generated = str(completion)
        
        # Normalize for comparison
        gen_clean = generated.lower().strip()
        exp_clean = expert.lower().strip()
        
        # Compute similarity (0.0 to 1.0)
        similarity = SequenceMatcher(None, gen_clean, exp_clean).ratio()
        
        # Scale reward: 0.0 (no match) to 1.0 (exact match)
        rewards.append(float(similarity))
    
    return rewards

# Test the reward function
test_completions = [[{"content": "Step 2: x = 7"}]]
test_expert = ["Step 2: x + 5 - 5 = 12 - 5, so x = 7."]
print(f"Test reward: {srl_reward_function([], test_completions, test_expert)}")

## 5. Configure GRPOTrainer

In [ ]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir="./srl_output",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    
    # GRPO specific
    num_generations=4,           # K rollouts per prompt
    max_completion_length=256,   # Step-level (short)
    temperature=1.0,
    
    # Memory optimization
    bf16=True,
    gradient_checkpointing=True,
    
    # vLLM
    use_vllm=True,
    vllm_gpu_memory_utilization=0.6,
    
    # Logging
    logging_steps=1,
    save_strategy="no",
    report_to="none",
)

## 6. Train with GRPO

In [ ]:
trainer = GRPOTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    reward_funcs=srl_reward_function,
    tokenizer=tokenizer,
)

print("Starting SRL Training...")
trainer.train()
print("Training complete!")

## 7. Test the Trained Model

In [ ]:
# Switch to inference mode
FastLanguageModel.for_inference(model)

# Test prompt
test_problem = "Problem: Solve 2x + 3 = 11 for x.\n\nStep 1: Subtract 3 from both sides."
messages = [
    {"role": "system", "content": SRL_INSTRUCTION},
    {"role": "user", "content": test_problem}
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Generate next step
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=128, temperature=0.7)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("=" * 50)
print("Test Problem:")
print(test_problem)
print("\nModel's Next Step:")
print(response.split(test_problem)[-1].strip())

## Next Steps

For full SRL training with larger datasets:
- Clone the [full repository](https://github.com/YOUR_USERNAME/Supervised-Reinforcement-Learning)
- Use provided dataset processing scripts
- Run Stage 2 (RLVR) for final answer correctness

**Key files:**
- `train_srl.py` - Full SRL training script
- `train_srl_rlvr.py` - Stage 2 RLVR training
- `srl_reward_function.py` - Advanced reward with semantic matching